# Run Match

In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [ ]:
from timeutils import Timestat
from master import MasterParams, MasterDBs, MasterPaths, MasterMetas
from musicdb import MusicDBIO
from gate import MusicDBGate
from pandas import DataFrame, Series, concat
import musicdb
from ioutils import HTMLIO, FileIO
from listUtils import getFlatList
import swifter
import dask.dataframe as dd
from match import getLevenshtein, MatchDB, MatchDBDataIO, AlbumReq, MatchID
mp     = MasterParams(verbose=True)
io     = FileIO()
gate   = MusicDBGate()

In [ ]:
baseDB    = "RateYourMusic"
baseDB    = "Genius"
baseReq   = {baseDB: AlbumReq(max=100, top=2000)}

compareDBs  = ["RateYourMusic", "LastFM", "Spotify", "Genius", "Discogs", "MusicBrainz", "Deezer", "MetalArchives"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz"]
compareDBs  = ["RateYourMusic", "Spotify", "Discogs", "MusicBrainz"]
compareReqs = {compareDB: AlbumReq(min=3) for compareDB in compareDBs if compareDB not in [baseDB]}
compareDBs  = list(compareReqs.keys())

albumReqs = {**baseReq, **compareReqs}
mediaTypes = ["Album", "SingleEP"]
mediaTypes = ["{0}Media".format(mediaType) for mediaType in mediaTypes]
reqs       = {"Media": mediaTypes, "Albums": albumReqs, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.8, "Medium": 2, "Tight": 1}}

print("Run Params:")
print("  ==> DBs:   [{0}] x {1}]".format(baseDB,list(compareReqs.keys())))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(reqs["Match"]))

In [ ]:
mdb = MatchDB(baseDB=baseDB, compareDBs=compareDBs, reqs=reqs)
mdb.match()

# Get Master IDs

In [ ]:
#io.save(idata=mdb, ifile="mdb.p")
mdb = io.get("mdb.p")

In [ ]:
mid = MatchID(baseDB, mdb, verbose=True)
mid.join()
mid.getMasterID()
mid.joinMaster()

In [ ]:
mid.mergeMaster()

# Other